# Laplace operator in JAX

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, vmap, jit

def laplace(func, x):
    """Compute the Laplace operator of the model output with respect to inputs."""
    grad_fn = grad(func)
    d2_dx2 = 0
    for i in range(x.shape[1]):
        d2_dx2 += vmap(grad(lambda xi: grad_fn(xi)[i]))(x)[:, i]
    return d2_dx2

def model(x):
    # x is (batch_size, input_dim)
    return jnp.sum(x**3, axis=-1) 

def solution(x):
    return jnp.sum(6 * x, axis=-1)

# Example usage

x = jax.random.uniform(jax.random.PRNGKey(0), (10, 2))  # 1000 samples, 3 dimensions
# x = jnp.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0], [9.0, 10.0]])
print(f"Input size: {x.shape}")
laplace_values = laplace(model, x)
solution_values = solution(x)
print(laplace_values)
print(solution_values)

Input size: (10, 2)
Second derivatives shape: (10,)
[11.557481   4.8057594  4.4123507  5.9980507  5.5066867  6.0638185
  7.2163963 11.002508   9.358538   3.2250316]
True
[11.557481   4.8057594  4.4123507  5.9980507  5.5066867  6.0638185
  7.2163963 11.002508   9.358538   3.2250316]


In [ ]:
def local_energy_batch(params, xs, model_apply):
    # xs: (batch, 1) or (batch,)
    xs_flat = xs.squeeze()  # shape (batch,)

    # psi(x) -> scalar
    def psi_fn(x):
        # ensure input has shape (1,) as model expects last-dim features
        return model_apply(params, x.reshape(1, 1)).squeeze()

    # second derivative per sample via AD
    d2psi_fn = jax.vmap(jax.jacfwd(jax.grad(psi_fn)))
    d2psi = d2psi_fn(xs_flat)  # shape (batch,)
    psi_vals = jax.vmap(lambda x: psi_fn(x))(xs_flat)  # shape (batch,)

    # avoid division by zero / small psi
    psi_safe = psi_vals + 1e-12

    kinetic = -0.5 * (d2psi / psi_safe)  # shape (batch,)
    potential = 0.5 * (xs_flat**2)  # shape (batch,)
    return (kinetic + potential).reshape(-1, 1)  # keep your (batch,1) convention